# IMPLEMENT MACHINE TRANSLATION USING TRANSFORMER

## Objective:
Train a Transformer model on French-English translation task using the fra-eng dataset.

## STEP 1 - Import libraries

### 1.1 Clone repository

In [ ]:
import shutil
import os

# Remove existing folder if it exists
if os.path.exists('DL_self_practice'):
    shutil.rmtree('DL_self_practice')

# Clone repository
!git clone https://github.com/ArrayPowerPlay/DL_self_practice.git

os.chdir('DL_self_practice')

### 1.2. Import libraries

In [ ]:
import os 
import torch
from utils import spy

### 1.3. Check if GPU is available

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(os.getcwd())

False


AssertionError: Torch not compiled with CUDA enabled

## STEP 2 - Load and preprocess data

In [12]:
# Load fra-eng dataset using MachineTranslation class
PATH = 'data/fra-eng/fra.txt'

data = spy.MachineTranslation(
    path=PATH,
    batch_size=128,              
    num_steps=25,                
    num_train=60000,           
    num_val=5000                 
)

print(f"Source vocabulary size: {len(data.src_vocab)}")
print(f"Target vocabulary size: {len(data.tgt_vocab)}")
print(f"Training examples: {data.num_train}")
print(f"Validation examples: {data.num_val}")

Source vocabulary size: 5017
Target vocabulary size: 8693
Training examples: 60000
Validation examples: 5000


## STEP 3 - Define Transformer model

### 3.1. Define some hyperparameters

In [ ]:
num_hiddens, num_blks, dropout = 512, 6, 0.15
ffn_num_hiddens, num_heads = 2048, 8

### 3.2. Define Transformer encoder

In [ ]:
encoder = spy.TransformerEncoder(
    len(data.src_vocab), num_hiddens, ffn_num_hiddens, 
    num_heads, num_blks, dropout
)

### 3.3. Define Transformer decoder

In [ ]:
decoder = spy.TransformerDecoder(
    len(data.tgt_vocab), num_hiddens, ffn_num_hiddens, 
    num_heads, num_blks, dropout
)

### 3.4. Define Transformer model

In [ ]:
model = spy.Seq2Seq(
    encoder, 
    decoder, 
    tgt_pad=data.tgt_vocab['<pad>'],
    lr=5e-4
) 

## STEP 4 - Training model

### 4.1. Create convenient functions

In [ ]:
import math

# Learning rate scheduler for training optimization
def get_lr_scheduler(optimizer, total_epochs):
    """Warm-up + cosine annealing learning rate scheduler"""
    def lr_lambda(epoch):
        warmup_epochs = 5
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        else:
            progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
            return 0.5 * (1 + math.cos(math.pi * progress))
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

### 4.2. Create a custom Transformer trainer class with learning rate scheduler

In [ ]:
class TransformerTrainer(spy.Trainer):
    def prepare_model(self, model):
        super().prepare_model(model)
        self.scheduler = get_lr_scheduler(self.optimizer, self.max_epochs)

### 4.3. Training Transformer model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
max_epochs = 60

trainer = TransformerTrainer(max_epochs=max_epochs, gradient_clip_val=1, device=device)
trainer.fit(model, data)

### 4.4. Save checkpoints

In [ ]:
torch.save({
    "model": model.state_dict(),
    "board": model.board,
    "optimizer": trainer.optimizer.state_dict(),
    "scheduler": trainer.scheduler.state_dict()
}, "machine_translation_checkpoint.pt")

print("Checkpoint saved!")

NameError: name 'model' is not defined

## STEP 5 - Test translation

In [ ]:
# Evaluate on validation data
model.eval()

for batch in list(data.val_dataloader())[:8]:
    batch = [b.to(trainer.device) for b in batch]
    src, tgt_input, src_valid_len, tgt_output = batch
    preds, _ = model.predict_step(batch, trainer.device, data.num_steps)
    
    for s, t, p in zip(src, tgt_output, preds):
        src_str = ' '.join(data.src_vocab.
                    to_tokens(s.cpu().tolist())).replace('<pad>', '').replace('<eos>', '').strip()
        
        ref_tokens = []
        for i in t:
            token = data.tgt_vocab.to_tokens(i.item())
            if token == '<eos>':
                break
            if token != '<pad>':
                ref_tokens.append(token)
        ref_str = ' '.join(ref_tokens).strip()
        
        pred_tokens = []
        for i in p:
            token = data.tgt_vocab.to_tokens(i.item())
            if token == '<eos>':
                break
            if token != '<pad>':
                pred_tokens.append(token)
        pred_str = ' '.join(pred_tokens).strip()
        
        bleu = spy.bleu(pred_str, ref_str, k=4)
        print(f'{src_str} => {pred_str} - bleu: {bleu:.3f}')